<a href="https://colab.research.google.com/github/ZinahO/AAI-520-Group-6-Multi-Agent-Financial-Analysis/blob/noslinn-routing-specialists/routing_specialist_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AAI-520 Final Team Project: Multi-Agent Financial Analysis System

## Routing, Specialist Agents, and Dynamic Tool Use

This section implements the routing workflow for the Investment
Research Agent. The router receives structured financial news from the
prompt-chaining workflow and directs each item to the Earnings, News, or
Market Specialist. The specialist agents then analyze the routed
information using appropriate financial tools and produce a structured
draft analysis for the Evaluator–Optimizer workflow.

In [ ]:
!pip install -q yfinance google-genai

In [ ]:
import json
import time

import pandas as pd
import yfinance as yf
from google import genai
from google.colab import userdata

In [3]:
gemini_api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(
    api_key=gemini_api_key,
    http_options={"timeout": 30000}
)

print("Gemini client connected successfully.")

Gemini client connected successfully.


In [6]:
last_llm_call = 0


def call_llm(prompt):
    """Send a prompt to Gemini with basic rate-limit handling."""
    global last_llm_call

    elapsed = time.time() - last_llm_call

    if elapsed < 15:
        time.sleep(15 - elapsed)

    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model="gemini-3.8-flash",
                contents=prompt
            )

            last_llm_call = time.time()
            return response.text

        except Exception:
            if attempt < 2:
                time.sleep(15)
            else:
                raise

In [7]:
test_response = call_llm(
    "Respond with exactly: Gemini routing connection test successful."
)

print(test_response)

Gemini routing connection test successful.


## Input from the Prompt-Chaining Workflow

The routing workflow receives the structured output produced by the
prompt-chaining workflow. Each processed article contains a title,
category, publication date, provider, and extracted key information.

The sample below follows the same input structure as Zinah's workflow
and allows the routing system to be developed and tested independently.

In [8]:
prompt_chain_output = {
    "symbol": "AAPL",
    "research_plan": (
        "1. Review recent company news.\n"
        "2. Examine earnings-related information.\n"
        "3. Analyze recent stock-market performance."
    ),
    "processed_news": [
        {
            "title": "Apple reports quarterly revenue and profit growth",
            "category": "Quarterly Financial Results",
            "pub_date": "2026-10-01",
            "provider": "Sample Financial News",
            "key_information": (
                "Apple reported increased quarterly revenue, earnings "
                "per share, and profit."
            ),
        },
        {
            "title": "Apple announces expansion of its product lineup",
            "category": "Company News",
            "pub_date": "2026-10-02",
            "provider": "Sample Technology News",
            "key_information": (
                "Apple announced new products and an expansion of its "
                "technology offerings."
            ),
        },
        {
            "title": "Apple shares rise alongside the technology sector",
            "category": "Stock Performance Analysis",
            "pub_date": "2026-10-03",
            "provider": "Sample Market News",
            "key_information": (
                "Apple shares increased as the broader technology sector "
                "and major market indexes moved higher."
            ),
        },
    ],
    "summary": (
        "Recent Apple coverage includes earnings results, company "
        "developments, and stock-market performance."
    ),
}

print(f"Symbol: {prompt_chain_output['symbol']}")
print(
    "Articles received:",
    len(prompt_chain_output["processed_news"]),
)

Symbol: AAPL
Articles received: 3


## Routing Workflow

The routing agent examines each article's title, category, and extracted
financial information. It directs the article to one of three specialists:

- **Earnings Specialist:** Revenue, profit, earnings, guidance, and financial results
- **News Specialist:** Products, leadership, partnerships, regulations, and company events
- **Market Specialist:** Stock prices, trading activity, analysts, and broader market movements

The router first uses transparent keyword scoring. If the content is
ambiguous or no keywords are detected, Gemini selects the appropriate
specialist dynamically.

In [9]:
VALID_ROUTES = ("earnings", "news", "market")

ROUTE_KEYWORDS = {
    "earnings": {
        "earnings",
        "revenue",
        "profit",
        "eps",
        "quarter",
        "quarterly",
        "financial results",
        "guidance",
        "margin",
        "income",
    },
    "news": {
        "product",
        "launch",
        "partnership",
        "acquisition",
        "lawsuit",
        "regulation",
        "leadership",
        "ceo",
        "operations",
        "expansion",
    },
    "market": {
        "shares",
        "stock",
        "market",
        "price",
        "trading",
        "investor",
        "index",
        "nasdaq",
        "analyst",
        "target",
        "volatility",
        "sector",
    },
}


def llm_route_article(article):
    """Use Gemini when keyword routing is ambiguous."""
    article_text = json.dumps(article, default=str)

    prompt = f"""
    Route the following financial article to exactly one specialist:

    earnings: financial results, revenue, profit, EPS, or guidance
    news: company announcements, products, leadership, or regulations
    market: stock price, trading, analysts, or broader market movement

    Return only one word: earnings, news, or market.

    Article:
    {article_text}
    """

    response = call_llm(prompt).strip().lower()

    for route in VALID_ROUTES:
        if route in response:
            return route

    return "news"


def route_article(article):
    """Select the most appropriate specialist for one article."""
    fields = [
        article.get("title", ""),
        article.get("category", ""),
        article.get("key_information", ""),
    ]
    article_text = " ".join(str(field) for field in fields).lower()

    scores = {
        route: sum(
            keyword in article_text
            for keyword in keywords
        )
        for route, keywords in ROUTE_KEYWORDS.items()
    }

    highest_score = max(scores.values())
    highest_routes = [
        route
        for route, score in scores.items()
        if score == highest_score
    ]

    if highest_score == 0 or len(highest_routes) > 1:
        selected_route = llm_route_article(article)
        method = "Gemini fallback"
    else:
        selected_route = highest_routes[0]
        method = "Keyword routing"

    return {
        "route": selected_route,
        "method": method,
        "scores": scores,
    }


def route_articles(prompt_output):
    """Route all processed articles to specialist queues."""
    routed_output = {
        "symbol": prompt_output["symbol"],
        "routes": {
            "earnings": [],
            "news": [],
            "market": [],
        },
        "routing_log": [],
    }

    for article in prompt_output["processed_news"]:
        decision = route_article(article)
        route = decision["route"]

        routed_article = article.copy()
        routed_article["assigned_specialist"] = route
        routed_output["routes"][route].append(routed_article)

        routed_output["routing_log"].append({
            "title": article.get("title", ""),
            **decision,
        })

    return routed_output

In [10]:
routed_output = route_articles(prompt_chain_output)

for route, articles in routed_output["routes"].items():
    print(f"{route.title()} Specialist: {len(articles)} article(s)")

print("\nRouting decisions:")

for decision in routed_output["routing_log"]:
    print(
        f"- {decision['title']} "
        f"-> {decision['route']} "
        f"({decision['method']})"
    )

Earnings Specialist: 1 article(s)
News Specialist: 1 article(s)
Market Specialist: 1 article(s)

Routing decisions:
- Apple reports quarterly revenue and profit growth -> earnings (Keyword routing)
- Apple announces expansion of its product lineup -> news (Keyword routing)
- Apple shares rise alongside the technology sector -> market (Keyword routing)


## Dynamic Financial Tools

Each specialist uses a different Yahoo Finance tool based on the route
selected by the routing agent:

- The Earnings Specialist retrieves quarterly financial metrics.
- The News Specialist retrieves company profile and industry context.
- The Market Specialist retrieves recent price and trading information.

A tool registry allows the system to select the appropriate function
dynamically instead of running every available tool.

In [11]:
def latest_financial_value(financials, metric):
    """Return the latest available value for a financial metric."""
    if financials.empty or metric not in financials.index:
        return None

    values = financials.loc[metric].dropna()

    if values.empty:
        return None

    return round(float(values.iloc[0]), 2)


def get_earnings_data(symbol):
    """Retrieve recent quarterly earnings metrics."""
    try:
        ticker = yf.Ticker(symbol)
        financials = ticker.quarterly_income_stmt

        return {
            "tool": "Yahoo Finance quarterly income statement",
            "symbol": symbol,
            "total_revenue": latest_financial_value(
                financials,
                "Total Revenue",
            ),
            "net_income": latest_financial_value(
                financials,
                "Net Income",
            ),
            "diluted_eps": latest_financial_value(
                financials,
                "Diluted EPS",
            ),
        }
    except Exception as error:
        return {
            "tool": "Yahoo Finance quarterly income statement",
            "symbol": symbol,
            "error": str(error),
        }


def get_company_profile(symbol):
    """Retrieve company and industry context."""
    try:
        ticker = yf.Ticker(symbol)
        information = ticker.info

        return {
            "tool": "Yahoo Finance company profile",
            "symbol": symbol,
            "company_name": information.get("longName"),
            "sector": information.get("sector"),
            "industry": information.get("industry"),
            "country": information.get("country"),
            "employee_count": information.get("fullTimeEmployees"),
        }
    except Exception as error:
        return {
            "tool": "Yahoo Finance company profile",
            "symbol": symbol,
            "error": str(error),
        }


def get_market_data(symbol):
    """Retrieve recent stock-price and market metrics."""
    try:
        ticker = yf.Ticker(symbol)
        history = ticker.history(
            period="1mo",
            auto_adjust=True,
        )

        if history.empty:
            return {
                "tool": "Yahoo Finance price history",
                "symbol": symbol,
                "error": "No market history was returned.",
            }

        closing_prices = history["Close"].dropna()
        daily_returns = closing_prices.pct_change().dropna()

        period_return = (
            (closing_prices.iloc[-1] / closing_prices.iloc[0]) - 1
        ) * 100

        annualized_volatility = (
            daily_returns.std() * (252 ** 0.5) * 100
        )

        return {
            "tool": "Yahoo Finance one-month price history",
            "symbol": symbol,
            "latest_close": round(float(closing_prices.iloc[-1]), 2),
            "one_month_return_pct": round(float(period_return), 2),
            "annualized_volatility_pct": round(
                float(annualized_volatility),
                2,
            ),
            "average_volume": round(
                float(history["Volume"].mean()),
                2,
            ),
        }
    except Exception as error:
        return {
            "tool": "Yahoo Finance price history",
            "symbol": symbol,
            "error": str(error),
        }


SPECIALIST_TOOLS = {
    "earnings": get_earnings_data,
    "news": get_company_profile,
    "market": get_market_data,
}


def use_specialist_tool(specialist, symbol):
    """Dynamically select and execute a specialist tool."""
    if specialist not in SPECIALIST_TOOLS:
        raise ValueError(
            f"Unknown specialist: {specialist}"
        )

    selected_tool = SPECIALIST_TOOLS[specialist]
    return selected_tool(symbol)

In [12]:
symbol = routed_output["symbol"]

for specialist in VALID_ROUTES:
    tool_result = use_specialist_tool(
        specialist,
        symbol,
    )

    print(f"\n{specialist.title()} Specialist Tool:")
    print(json.dumps(tool_result, indent=2, default=str))


Earnings Specialist Tool:
{
  "tool": "Yahoo Finance quarterly income statement",
  "symbol": "AAPL",
  "total_revenue": 109417000000.0,
  "net_income": 29789000000.0,
  "diluted_eps": 2.02
}

News Specialist Tool:
{
  "tool": "Yahoo Finance company profile",
  "symbol": "AAPL",
  "company_name": "Apple Inc.",
  "sector": "Technology",
  "industry": "Consumer Electronics",
  "country": "United States",
  "employee_count": 150000
}

Market Specialist Tool:
{
  "tool": "Yahoo Finance one-month price history",
  "symbol": "AAPL",
  "latest_close": 333.69,
  "one_month_return_pct": 1.67,
  "annualized_volatility_pct": 23.09,
  "average_volume": 41995709.52
}


## Specialist Agents

Each specialist receives only the articles assigned to its area and
dynamically calls the appropriate Yahoo Finance tool. Gemini combines
the article information with the retrieved financial data to produce a
focused analysis.

Each specialist reports key findings, supporting evidence, and risks or
limitations. The resulting analyses will be passed to the team's
Evaluator–Optimizer workflow.

In [13]:
SPECIALIST_INSTRUCTIONS = {
    "earnings": (
        "Analyze revenue, profitability, earnings per share, and "
        "financial performance. Compare the article claims with the "
        "available quarterly financial metrics."
    ),
    "news": (
        "Analyze the strategic importance of company announcements. "
        "Consider the company's sector, industry, operations, and "
        "potential business implications."
    ),
    "market": (
        "Analyze recent stock performance, return, volatility, and "
        "trading activity. Do not claim that a news event caused a "
        "price movement unless the provided evidence supports it."
    ),
}


def run_specialist_agent(specialist, symbol, articles):
    """Run one specialist agent with its dynamically selected tool."""
    tool_data = use_specialist_tool(
        specialist,
        symbol,
    )

    prompt = f"""
    You are the {specialist.title()} Specialist for an autonomous
    investment research system.

    Your responsibility:
    {SPECIALIST_INSTRUCTIONS[specialist]}

    Stock symbol:
    {symbol}

    Routed articles:
    {json.dumps(articles, indent=2, default=str)}

    Data retrieved by your financial tool:
    {json.dumps(tool_data, indent=2, default=str)}

    Produce a concise financial analysis using only the supplied
    evidence.

    Use these headings:
    Key Findings
    Supporting Evidence
    Risks and Limitations

    Clearly distinguish reported facts from your interpretation.
    Do not provide a buy or sell recommendation.
    """

    analysis = call_llm(prompt).strip()

    return {
        "specialist": specialist,
        "symbol": symbol,
        "articles_reviewed": len(articles),
        "tool_used": tool_data.get("tool"),
        "tool_data": tool_data,
        "analysis": analysis,
    }


def earnings_specialist(symbol, articles):
    """Analyze earnings-related content."""
    return run_specialist_agent(
        "earnings",
        symbol,
        articles,
    )


def news_specialist(symbol, articles):
    """Analyze company-news content."""
    return run_specialist_agent(
        "news",
        symbol,
        articles,
    )


def market_specialist(symbol, articles):
    """Analyze market-related content."""
    return run_specialist_agent(
        "market",
        symbol,
        articles,
    )


SPECIALIST_AGENTS = {
    "earnings": earnings_specialist,
    "news": news_specialist,
    "market": market_specialist,
}

In [15]:
def run_specialist_workflow(routed_data):
    """Execute the appropriate agent for each routed article group."""
    symbol = routed_data["symbol"]
    specialist_results = {}

    for specialist, articles in routed_data["routes"].items():
        if not articles:
            specialist_results[specialist] = {
                "specialist": specialist,
                "symbol": symbol,
                "articles_reviewed": 0,
                "status": "Skipped because no articles were routed here.",
            }
            continue

        selected_agent = SPECIALIST_AGENTS[specialist]

        specialist_results[specialist] = selected_agent(
            symbol,
            articles,
        )

    return {
        "symbol": symbol,
        "specialist_results": specialist_results,
    }


specialist_output = run_specialist_workflow(routed_output)

In [16]:
for specialist, result in (
    specialist_output["specialist_results"].items()
):
    print(f"\n{'=' * 60}")
    print(f"{specialist.upper()} SPECIALIST")
    print(f"Articles reviewed: {result['articles_reviewed']}")
    print(f"Tool used: {result.get('tool_used', 'None')}")
    print("-" * 60)
    print(result.get("analysis", result.get("status")))


EARNINGS SPECIALIST
Articles reviewed: 1
Tool used: Yahoo Finance quarterly income statement
------------------------------------------------------------
### Key Findings
* **Current Financial Metrics (Reported Fact):** For the reported quarter, Apple Inc. (AAPL) recorded total revenue of $109.417 billion, net income of $29.789 billion, and diluted earnings per share (EPS) of $2.02.
* **Profitability Assessment (Interpretation):** Based on the reported figures, Apple achieved a net profit margin of approximately 27.23% ($29.789 billion net income divided by $109.417 billion total revenue), reflecting strong baseline conversion of revenue to bottom-line earnings.
* **Alignment with News Claims (Interpretation):** While the article reports that Apple achieved growth across revenue, profit, and earnings per share, the financial tool provides only a single-period snapshot. Consequently, the claim of "growth" cannot be verified from the dataset provided, as comparative prior-period metrics

## Gemini Fallback Routing Test

The standard routing test used articles containing recognizable financial
keywords. This additional test uses deliberately ambiguous content with
no matching routing keywords. The router should dynamically ask Gemini
to select the appropriate specialist.

In [18]:
ambiguous_article = {
    "title": "Apple provides an update to stakeholders",
    "category": "Corporate Update",
    "pub_date": "2026-10-04",
    "provider": "Sample Business Source",
    "key_information": (
        "Apple discussed recent developments and future priorities "
        "with stakeholders."
    ),
}

fallback_decision = route_article(ambiguous_article)

print("Ambiguous article routing test:")
print(json.dumps(fallback_decision, indent=2))

assert fallback_decision["method"] == "Gemini fallback"
assert fallback_decision["route"] in VALID_ROUTES

print("\nGemini fallback test passed.")

Ambiguous article routing test:
{
  "route": "news",
  "method": "Gemini fallback",
  "scores": {
    "earnings": 0,
    "news": 0,
    "market": 0
  }
}

Gemini fallback test passed.


## End-to-End Routing and Specialist Workflow

This function connects the routing agent, dynamic tool selection, and
specialist agents into one reusable workflow. It accepts the structured
output from the prompt-chaining workflow and produces a structured
dictionary for the Evaluator–Optimizer workflow.

In [19]:
def run_routing_specialist_workflow(prompt_output):
    """Run routing, tool selection, and specialist analysis."""
    routed_data = route_articles(prompt_output)
    specialist_data = run_specialist_workflow(routed_data)

    return {
        "symbol": prompt_output["symbol"],
        "research_plan": prompt_output.get("research_plan"),
        "source_summary": prompt_output.get("summary"),
        "routing_log": routed_data["routing_log"],
        "routed_articles": routed_data["routes"],
        "specialist_results": specialist_data[
            "specialist_results"
        ],
        "workflow_status": "completed",
    }

In [20]:
routing_specialist_output = {
    "symbol": prompt_chain_output["symbol"],
    "research_plan": prompt_chain_output.get("research_plan"),
    "source_summary": prompt_chain_output.get("summary"),
    "routing_log": routed_output["routing_log"],
    "routed_articles": routed_output["routes"],
    "specialist_results": specialist_output[
        "specialist_results"
    ],
    "workflow_status": "completed",
}

print("Workflow status:", routing_specialist_output["workflow_status"])
print("Symbol:", routing_specialist_output["symbol"])
print(
    "Specialists completed:",
    list(routing_specialist_output["specialist_results"].keys()),
)

Workflow status: completed
Symbol: AAPL
Specialists completed: ['earnings', 'news', 'market']


In [21]:
# Run later during final integration testing.
# full_test_output = run_routing_specialist_workflow(
#     prompt_chain_output
# )